# Assignment 2.1: Advanced Genetic Algorithm Operators

## 🎯 Learning Objectives

In this assignment, you will:
- Master advanced selection methods (rank-based, SUS)
- Implement sophisticated crossover operators (arithmetic, BLX-α)
- Create adaptive mutation strategies
- Understand elitism and replacement strategies
- Compare operator performance on benchmark problems
- Build a production-grade GA with advanced operators

This assignment takes your GA skills to the **next level** with state-of-the-art operators!

---

## 📦 1. Setup and Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys

# Import basic GA utilities from Tutorial 1
sys.path.append('../GA_Tutorial_1_Basics')
from ga_utils_basics import sphere_function, rastrigin_function

# Import visualization toolkit
sys.path.append('../ga_toolkit')
from visualization import plot_convergence, plot_convergence_comparison

np.random.seed(42)

print("✅ Libraries imported successfully!")

---

## 🎯 2. Advanced Selection Methods

### 2.1 - Rank-Based Selection

**Problem with fitness-proportionate selection:**
- Premature convergence when one individual is much better
- Slow convergence when fitness values are similar

**Solution: Rank-based selection**
- Selection probability based on **rank**, not raw fitness
- More stable across different fitness landscapes

**Formula:**
$$P(i) = \frac{2 - SP + 2(SP-1) \frac{rank_i}{N-1}}{N}$$

where:
- $SP$ = selection pressure (1.0-2.0)
- $N$ = population size
- $rank_i$ = rank of individual i (0 = worst, N-1 = best)

### Exercise 1: Implement Rank-Based Selection

**Instructions:**
1. Rank individuals by fitness (worst = 0, best = N-1)
2. Calculate selection probabilities using the formula above
3. Select parents using probabilities

In [ ]:
def rank_based_selection(population, fitness, n_parents, selection_pressure=2.0):
    """
    Rank-based selection.
    
    Arguments:
    population -- numpy array (pop_size, n_variables)
    fitness -- numpy array (pop_size,)
    n_parents -- number of parents to select
    selection_pressure -- SP parameter (1.0-2.0), higher = more selective
    
    Returns:
    parents -- numpy array (n_parents, n_variables)
    """
    
    pop_size = len(population)
    
    ### START CODE HERE ### (≈ 8-10 lines)
    
    # Step 1: Rank individuals (0 = worst, pop_size-1 = best)
    # Hint: Use np.argsort twice to get ranks
    ranks = np.argsort(np.argsort(fitness))
    
    # Step 2: Calculate selection probabilities
    probabilities = np.zeros(pop_size)
    for i in range(pop_size):
        probabilities[i] = (2 - selection_pressure + 
                          2 * (selection_pressure - 1) * ranks[i] / (pop_size - 1)) / pop_size
    
    # Step 3: Normalize probabilities
    probabilities = probabilities / np.sum(probabilities)
    
    # Step 4: Select parents
    selected_indices = np.random.choice(pop_size, size=n_parents, p=probabilities, replace=True)
    parents = population[selected_indices]
    
    ### END CODE HERE ###
    
    return parents

In [ ]:
# Test your implementation
print("Testing rank-based selection:")
print("=" * 70)

test_pop = np.array([
    [1.0, 1.0],  # fitness = -2.0
    [0.0, 0.0],  # fitness = 0.0 (best)
    [2.0, 2.0],  # fitness = -8.0 (worst)
    [0.5, 0.5],  # fitness = -0.5
])

test_fitness = np.array([-2.0, 0.0, -8.0, -0.5])

# Select 100 parents to see distribution
np.random.seed(42)
parents = rank_based_selection(test_pop, test_fitness, n_parents=100, selection_pressure=2.0)

# Count how many times each individual was selected
counts = []
for i in range(len(test_pop)):
    count = np.sum(np.all(parents == test_pop[i], axis=1))
    counts.append(count)

print("\nSelection distribution (100 parents):")
for i, (fit, count) in enumerate(zip(test_fitness, counts)):
    rank = np.argsort(np.argsort(test_fitness))[i]
    print(f"  Individual {i} (rank={rank}, fitness={fit:.1f}): selected {count} times")

# Best individual (rank 3) should be selected most often
best_idx = np.argmax(test_fitness)
assert counts[best_idx] > 30, "Best individual should be selected most often"

print("\n✅ Test passed! Best individual selected most frequently.")

**Expected Output:**
```
Selection distribution (100 parents):
  Individual 0 (rank=1, fitness=-2.0): selected ~20 times
  Individual 1 (rank=3, fitness=0.0): selected ~40 times  ← Best
  Individual 2 (rank=0, fitness=-8.0): selected ~10 times  ← Worst
  Individual 3 (rank=2, fitness=-0.5): selected ~30 times

✅ Test passed!
```

### 2.2 - Stochastic Universal Sampling (SUS)

**Problem with roulette wheel selection:**
- High variance in number of offspring
- Poor individuals might not be selected at all
- Good individuals might dominate

**SUS Solution:**
- Uses **equally spaced pointers** instead of multiple random spins
- Lower variance, more even sampling
- One random start point, then deterministic spacing

### Exercise 2: Implement Stochastic Universal Sampling

**Algorithm:**
1. Calculate cumulative fitness
2. Pointer distance = total_fitness / n_parents
3. Random start in [0, pointer_distance]
4. Place equally spaced pointers
5. Select individual at each pointer

In [ ]:
def stochastic_universal_sampling(population, fitness, n_parents):
    """
    Stochastic Universal Sampling (SUS).
    
    Arguments:
    population -- numpy array (pop_size, n_variables)
    fitness -- numpy array (pop_size,)
    n_parents -- number of parents to select
    
    Returns:
    parents -- numpy array (n_parents, n_variables)
    """
    
    ### START CODE HERE ### (≈ 12-15 lines)
    
    # Handle negative fitness (shift to positive)
    min_fitness = np.min(fitness)
    if min_fitness < 0:
        adjusted_fitness = fitness - min_fitness + 1e-10
    else:
        adjusted_fitness = fitness + 1e-10
    
    # Calculate cumulative fitness
    total_fitness = np.sum(adjusted_fitness)
    cumulative_fitness = np.cumsum(adjusted_fitness)
    
    # Pointer distance
    pointer_distance = total_fitness / n_parents
    
    # Random start point
    start = np.random.uniform(0, pointer_distance)
    
    # Generate pointers
    pointers = [start + i * pointer_distance for i in range(n_parents)]
    
    # Select individuals
    parents = []
    for pointer in pointers:
        for i, cum_fit in enumerate(cumulative_fitness):
            if pointer <= cum_fit:
                parents.append(population[i].copy())
                break
    
    ### END CODE HERE ###
    
    return np.array(parents)

In [ ]:
# Test your implementation
print("Testing Stochastic Universal Sampling:")
print("=" * 70)

test_pop = np.array([
    [1.0, 1.0],  # fitness = -2.0
    [0.0, 0.0],  # fitness = 0.0 (best)
    [2.0, 2.0],  # fitness = -8.0 (worst)
    [0.5, 0.5],  # fitness = -0.5
])

test_fitness = np.array([-2.0, 0.0, -8.0, -0.5])

np.random.seed(42)
parents = stochastic_universal_sampling(test_pop, test_fitness, n_parents=10)

print(f"\nSelected {len(parents)} parents")
print(f"Parents shape: {parents.shape}")

# Should select exactly n_parents
assert parents.shape == (10, 2), f"Expected shape (10, 2), got {parents.shape}"

print("\n✅ Test passed!")

---

## 🧬 3. Advanced Crossover Operators

### 3.1 - Arithmetic Crossover

**Arithmetic crossover** creates offspring as weighted averages of parents:

$$\text{child}_1 = \alpha \cdot \text{parent}_1 + (1-\alpha) \cdot \text{parent}_2$$
$$\text{child}_2 = (1-\alpha) \cdot \text{parent}_1 + \alpha \cdot \text{parent}_2$$

**Advantages:**
- Always produces valid offspring (within parent bounds)
- Good for continuous optimization
- Adjustable exploration vs exploitation

### Exercise 3: Implement Arithmetic Crossover

In [ ]:
def arithmetic_crossover(parent1, parent2, alpha=0.5):
    """
    Arithmetic crossover for real-valued chromosomes.
    
    Arguments:
    parent1, parent2 -- numpy arrays (n_variables,)
    alpha -- blending parameter (0-1), default 0.5 for equal blend
    
    Returns:
    child1, child2 -- numpy arrays (n_variables,)
    """
    
    ### START CODE HERE ### (≈ 2 lines)
    child1 = alpha * parent1 + (1 - alpha) * parent2
    child2 = (1 - alpha) * parent1 + alpha * parent2
    ### END CODE HERE ###
    
    return child1, child2

In [ ]:
# Test your implementation
print("Testing arithmetic crossover:")
print("=" * 70)

parent1 = np.array([10.0, 20.0, 30.0])
parent2 = np.array([0.0, 0.0, 0.0])

# Test with alpha = 0.5 (equal blend)
child1, child2 = arithmetic_crossover(parent1, parent2, alpha=0.5)

print(f"\nParent 1: {parent1}")
print(f"Parent 2: {parent2}")
print(f"Child 1 (α=0.5): {child1}")
print(f"Child 2 (α=0.5): {child2}")

# With alpha=0.5, children should be [5, 10, 15]
expected = np.array([5.0, 10.0, 15.0])
assert np.allclose(child1, expected), f"Expected {expected}, got {child1}"
assert np.allclose(child2, expected), f"Expected {expected}, got {child2}"

# Test with alpha = 0.7
child1, child2 = arithmetic_crossover(parent1, parent2, alpha=0.7)
print(f"\nChild 1 (α=0.7): {child1}")
print(f"Child 2 (α=0.7): {child2}")

print("\n✅ All tests passed!")

### 3.2 - BLX-α Crossover (Blend Crossover)

**BLX-α** creates offspring in an **extended range** around the parents:

For each gene:
1. Find min and max values from parents
2. Calculate interval $I = \max - \min$
3. Extended range: $[\min - \alpha \cdot I, \max + \alpha \cdot I]$
4. Sample uniformly from extended range

**Why BLX-α?**
- Can explore beyond parent values
- Good for escaping local optima
- Used in successful GAs like CHC

### Exercise 4: Implement BLX-α Crossover

In [ ]:
def blx_alpha_crossover(parent1, parent2, alpha=0.5, bounds=None):
    """
    BLX-α (Blend Crossover) for real-valued chromosomes.
    
    Arguments:
    parent1, parent2 -- numpy arrays (n_variables,)
    alpha -- extension parameter (typically 0.5)
    bounds -- optional tuple (lower, upper) to clip values
    
    Returns:
    child1, child2 -- numpy arrays (n_variables,)
    """
    
    child1 = np.zeros_like(parent1)
    child2 = np.zeros_like(parent2)
    
    ### START CODE HERE ### (≈ 8-10 lines)
    
    for i in range(len(parent1)):
        # Find min and max
        min_val = min(parent1[i], parent2[i])
        max_val = max(parent1[i], parent2[i])
        interval = max_val - min_val
        
        # Extended range
        lower = min_val - alpha * interval
        upper = max_val + alpha * interval
        
        # Sample from extended range
        child1[i] = np.random.uniform(lower, upper)
        child2[i] = np.random.uniform(lower, upper)
    
    # Clip to bounds if provided
    if bounds is not None:
        child1 = np.clip(child1, bounds[0], bounds[1])
        child2 = np.clip(child2, bounds[0], bounds[1])
    
    ### END CODE HERE ###
    
    return child1, child2

In [ ]:
# Test your implementation
print("Testing BLX-α crossover:")
print("=" * 70)

np.random.seed(42)

parent1 = np.array([2.0, 4.0, 6.0])
parent2 = np.array([0.0, 0.0, 0.0])

child1, child2 = blx_alpha_crossover(parent1, parent2, alpha=0.5)

print(f"\nParent 1: {parent1}")
print(f"Parent 2: {parent2}")
print(f"Child 1: {child1}")
print(f"Child 2: {child2}")

# For gene 0: min=0, max=2, interval=2, range=[-1, 3]
# Child values should be in extended range
assert -1 <= child1[0] <= 3, f"Child1[0] = {child1[0]} not in range [-1, 3]"
assert -1 <= child2[0] <= 3, f"Child2[0] = {child2[0]} not in range [-1, 3]"

# Test with bounds
child1, child2 = blx_alpha_crossover(parent1, parent2, alpha=0.5, bounds=(0, 10))
print(f"\nWith bounds [0, 10]:")
print(f"Child 1: {child1}")
print(f"Child 2: {child2}")

# Should be clipped to [0, 10]
assert np.all(child1 >= 0) and np.all(child1 <= 10), "Child1 violates bounds"
assert np.all(child2 >= 0) and np.all(child2 <= 10), "Child2 violates bounds"

print("\n✅ All tests passed!")

---

## 🎲 4. Advanced Mutation Operators

### 4.1 - Polynomial Mutation

**Polynomial mutation** is the mutation operator used in NSGA-II:

- Distribution similar to polynomial distribution
- Parameter $\eta$ controls spread (higher = closer to original)
- Commonly used in modern GAs

### Exercise 5: Implement Polynomial Mutation

**Instructions:**
- Mutate each gene with probability `mutation_rate`
- Use polynomial distribution formula
- Clip to bounds

In [ ]:
def polynomial_mutation(chromosome, mutation_rate, bounds, eta=20):
    """
    Polynomial mutation (used in NSGA-II).
    
    Arguments:
    chromosome -- numpy array (n_variables,)
    mutation_rate -- probability of mutating each gene
    bounds -- tuple (lower, upper)
    eta -- distribution index (higher = less spread)
    
    Returns:
    mutated -- numpy array (n_variables,)
    """
    
    mutated = chromosome.copy()
    lower, upper = bounds
    
    ### START CODE HERE ### (≈ 12-15 lines)
    
    for i in range(len(mutated)):
        if np.random.rand() < mutation_rate:
            u = np.random.rand()
            delta_l = (mutated[i] - lower) / (upper - lower)
            delta_u = (upper - mutated[i]) / (upper - lower)
            
            # Calculate delta_q
            if u < 0.5:
                delta_q = (2 * u) ** (1.0 / (eta + 1)) - 1.0
            else:
                delta_q = 1.0 - (2 * (1 - u)) ** (1.0 / (eta + 1))
            
            # Apply mutation
            mutated[i] = mutated[i] + delta_q * (upper - lower)
            
            # Clip to bounds
            mutated[i] = np.clip(mutated[i], lower, upper)
    
    ### END CODE HERE ###
    
    return mutated

In [ ]:
# Test your implementation
print("Testing polynomial mutation:")
print("=" * 70)

np.random.seed(42)

original = np.array([5.0, 5.0, 5.0, 5.0, 5.0])
mutated = polynomial_mutation(original, mutation_rate=0.5, bounds=(0, 10), eta=20)

print(f"\nOriginal: {original}")
print(f"Mutated:  {mutated}")
print(f"\nGenes mutated: {np.sum(original != mutated)}")

# Should be within bounds
assert np.all(mutated >= 0) and np.all(mutated <= 10), "Mutated values out of bounds"

# With mutation_rate=0.5, expect some mutations (not all, not none)
n_mutated = np.sum(original != mutated)
print(f"Mutation rate: {n_mutated / len(original) * 100:.0f}%")

print("\n✅ Test passed!")

### 4.2 - Adaptive Mutation

**Problem with fixed mutation rate:**
- Too high early: disrupts good solutions
- Too low late: can't escape local optima

**Solution: Adaptive mutation**
- Decrease mutation strength over generations
- High exploration early, fine-tuning late

### Exercise 6: Implement Adaptive Mutation

In [ ]:
def adaptive_mutation(chromosome, generation, max_generations, bounds,
                     initial_rate=0.3, final_rate=0.01):
    """
    Adaptive mutation that decreases over time.
    
    Arguments:
    chromosome -- numpy array (n_variables,)
    generation -- current generation
    max_generations -- total generations
    bounds -- tuple (lower, upper)
    initial_rate -- starting mutation rate
    final_rate -- ending mutation rate
    
    Returns:
    mutated -- numpy array (n_variables,)
    """
    
    ### START CODE HERE ### (≈ 12-15 lines)
    
    # Linear decrease in mutation rate
    current_rate = initial_rate - (initial_rate - final_rate) * generation / max_generations
    
    # Apply Gaussian mutation with adaptive rate
    mutated = chromosome.copy()
    lower, upper = bounds
    mutation_range = upper - lower
    
    for i in range(len(mutated)):
        if np.random.rand() < current_rate:
            # Decrease sigma over time
            sigma = (1 - generation / max_generations) * 0.2
            noise = np.random.normal(0, sigma * mutation_range)
            mutated[i] += noise
            mutated[i] = np.clip(mutated[i], lower, upper)
    
    ### END CODE HERE ###
    
    return mutated

In [ ]:
# Test your implementation
print("Testing adaptive mutation:")
print("=" * 70)

np.random.seed(42)

original = np.array([5.0] * 10)

# Test at different generations
gen_0 = adaptive_mutation(original, generation=0, max_generations=100, bounds=(0, 10))
gen_50 = adaptive_mutation(original, generation=50, max_generations=100, bounds=(0, 10))
gen_99 = adaptive_mutation(original, generation=99, max_generations=100, bounds=(0, 10))

print("\nOriginal:      ", original)
print("Gen 0 mutated: ", gen_0)
print("Gen 50 mutated:", gen_50)
print("Gen 99 mutated:", gen_99)

# Calculate average change
change_0 = np.mean(np.abs(gen_0 - original))
change_50 = np.mean(np.abs(gen_50 - original))
change_99 = np.mean(np.abs(gen_99 - original))

print(f"\nAverage change gen 0:  {change_0:.3f}")
print(f"Average change gen 50: {change_50:.3f}")
print(f"Average change gen 99: {change_99:.3f}")

# Change should decrease over generations
print(f"\nMutation strength decreases: {change_0 > change_50 > change_99}")

print("\n✅ Test passed!")

---

## 🏆 5. Elitism and Replacement Strategies

### 5.1 - Elitist Replacement

**Elitism**: Always keep the best individuals from previous generation

**Benefits:**
- Guarantees monotonic improvement (best never gets worse)
- Faster convergence
- Prevents loss of good solutions

**Drawback:**
- Can reduce diversity
- May cause premature convergence

### Exercise 7: Implement Elitist Replacement

In [ ]:
def elitist_replacement(old_population, old_fitness, new_population, new_fitness, n_elites=2):
    """
    Replace worst individuals in new population with best from old.
    
    Arguments:
    old_population -- previous generation (pop_size, n_variables)
    old_fitness -- fitness of old population (pop_size,)
    new_population -- new generation (pop_size, n_variables)
    new_fitness -- fitness of new population (pop_size,)
    n_elites -- number of elites to preserve
    
    Returns:
    population -- combined population (pop_size, n_variables)
    fitness -- combined fitness (pop_size,)
    """
    
    ### START CODE HERE ### (≈ 8-10 lines)
    
    # Find indices of best individuals in old population
    elite_indices = np.argsort(old_fitness)[-n_elites:]
    
    # Find indices of worst individuals in new population
    worst_indices = np.argsort(new_fitness)[:n_elites]
    
    # Replace worst in new with best from old
    population = new_population.copy()
    fitness = new_fitness.copy()
    
    for i, (elite_idx, worst_idx) in enumerate(zip(elite_indices, worst_indices)):
        population[worst_idx] = old_population[elite_idx].copy()
        fitness[worst_idx] = old_fitness[elite_idx]
    
    ### END CODE HERE ###
    
    return population, fitness

In [ ]:
# Test your implementation
print("Testing elitist replacement:")
print("=" * 70)

# Old population with one very good individual
old_pop = np.array([
    [1.0, 1.0],  # fitness = -2
    [0.0, 0.0],  # fitness = 0 (BEST)
    [2.0, 2.0],  # fitness = -8
])
old_fit = np.array([-2.0, 0.0, -8.0])

# New population without the best
new_pop = np.array([
    [1.5, 1.5],  # fitness = -4.5
    [1.0, 1.0],  # fitness = -2
    [2.5, 2.5],  # fitness = -12.5 (WORST)
])
new_fit = np.array([-4.5, -2.0, -12.5])

# Apply elitism (keep top 1)
combined_pop, combined_fit = elitist_replacement(
    old_pop, old_fit, new_pop, new_fit, n_elites=1
)

print("\nOld population best fitness:", np.max(old_fit))
print("New population best fitness:", np.max(new_fit))
print("Combined best fitness:      ", np.max(combined_fit))

# Best from old should be preserved
assert np.max(combined_fit) == 0.0, "Best individual not preserved!"
assert np.any(np.all(combined_pop == [0.0, 0.0], axis=1)), "Elite not in combined population!"

print("\n✅ Test passed! Elite preserved.")

---

## 🚀 6. Complete Advanced GA

Now let's build a complete GA using all advanced operators!

In [ ]:
def advanced_genetic_algorithm(fitness_func, bounds, pop_size=100, max_generations=200,
                              mutation_rate=0.1, crossover_rate=0.9, n_elites=2):
    """
    Advanced GA with rank selection, BLX-α crossover, adaptive mutation, and elitism.
    """
    
    n_variables = len(bounds)
    
    # Initialize
    population = np.random.uniform(
        [b[0] for b in bounds],
        [b[1] for b in bounds],
        size=(pop_size, n_variables)
    )
    
    history = {'best_fitness': [], 'mean_fitness': []}
    
    for generation in range(max_generations):
        # Evaluate (minimize, so negate)
        fitness = -np.array([fitness_func(ind) for ind in population])
        
        # Track history
        best_fit = np.max(fitness)
        history['best_fitness'].append(best_fit)
        history['mean_fitness'].append(np.mean(fitness))
        
        if generation % 50 == 0:
            print(f"Gen {generation:3d} | Best: {-best_fit:8.4f} | Mean: {-np.mean(fitness):8.4f}")
        
        # Selection (rank-based)
        parents = rank_based_selection(population, fitness, pop_size, selection_pressure=2.0)
        
        # Crossover and Mutation
        offspring = []
        for i in range(0, pop_size, 2):
            p1 = parents[i]
            p2 = parents[min(i+1, pop_size-1)]
            
            # BLX-α crossover
            if np.random.rand() < crossover_rate:
                c1, c2 = blx_alpha_crossover(p1, p2, alpha=0.5, bounds=(bounds[0][0], bounds[0][1]))
            else:
                c1, c2 = p1.copy(), p2.copy()
            
            # Adaptive mutation
            c1 = adaptive_mutation(c1, generation, max_generations, 
                                  bounds=(bounds[0][0], bounds[0][1]))
            c2 = adaptive_mutation(c2, generation, max_generations,
                                  bounds=(bounds[0][0], bounds[0][1]))
            
            offspring.extend([c1, c2])
        
        # Create new population
        new_population = np.array(offspring[:pop_size])
        new_fitness = -np.array([fitness_func(ind) for ind in new_population])
        
        # Elitist replacement
        population, fitness = elitist_replacement(
            population, fitness, new_population, new_fitness, n_elites=n_elites
        )
    
    # Final evaluation
    best_idx = np.argmax(fitness)
    best_solution = population[best_idx]
    best_fitness = fitness[best_idx]
    
    return best_solution, best_fitness, history

### 6.1 - Test on Rastrigin Function

Rastrigin is a **multimodal** function with many local optima - perfect for testing advanced operators!

In [ ]:
print("\n" + "="*70)
print("ADVANCED GA ON RASTRIGIN FUNCTION")
print("="*70)

bounds = [(-5.12, 5.12)] * 5

best_sol, best_fit, history = advanced_genetic_algorithm(
    fitness_func=rastrigin_function,
    bounds=bounds,
    pop_size=100,
    max_generations=200,
    n_elites=2
)

print("\n" + "="*70)
print("RESULTS")
print("="*70)
print(f"Best solution: {best_sol}")
print(f"Best fitness: {-best_fit:.6f}")
print(f"Distance from optimum: {np.linalg.norm(best_sol):.6f}")
print("\n✅ Advanced GA completed!")

In [ ]:
# Visualize convergence
plot_convergence(history, title="Advanced GA on Rastrigin Function")
plt.show()

---

## 📊 7. Compare Operators

Let's compare different selection methods!

In [ ]:
print("Comparing selection methods...")
print("="*70)

# Run multiple trials
n_trials = 3
histories = {}

for trial in range(1, n_trials + 1):
    np.random.seed(trial * 42)
    print(f"\nTrial {trial}/{n_trials}...", end=" ")
    
    _, _, hist = advanced_genetic_algorithm(
        fitness_func=sphere_function,
        bounds=[(-5, 5)] * 5,
        pop_size=50,
        max_generations=100,
        n_elites=2
    )
    
    histories[f'Trial {trial}'] = hist
    print(f"Done! Final: {-hist['best_fitness'][-1]:.6f}")

# Plot comparison
plot_convergence_comparison(
    histories,
    title="Advanced GA - Multiple Trials"
)
plt.show()

print("\n✅ Comparison complete!")

---

## 💡 8. Key Insights

### What You Learned:

1. **Advanced Selection Methods**
   - Rank-based: More stable than fitness-proportionate
   - SUS: Lower variance than roulette wheel
   - Both handle negative fitness better

2. **Sophisticated Crossover**
   - Arithmetic: Simple, always valid offspring
   - BLX-α: Can explore beyond parents
   - Different operators for different problems

3. **Adaptive Strategies**
   - Decrease mutation over time
   - High exploration early → fine-tuning late
   - Better convergence

4. **Elitism**
   - Guarantees monotonic improvement
   - Small elitism (2-5%) often optimal
   - Too much → premature convergence

### Performance Improvements:

Advanced operators typically provide:
- **20-40% faster convergence**
- **10-30% better final solutions**
- **More consistent results** (lower variance)

### When to Use What:

| Problem Type | Best Selection | Best Crossover | Best Mutation |
|--------------|---------------|----------------|---------------|
| Continuous, unimodal | Rank-based | Arithmetic | Adaptive |
| Continuous, multimodal | SUS | BLX-α | Polynomial |
| Mixed difficulties | Tournament | Blend | Adaptive |

---

## 🎯 9. Challenge Exercise

**Challenge:** Implement and compare different operator combinations!

Try:
1. Rank-based + Arithmetic + Polynomial mutation
2. SUS + BLX-α + Adaptive mutation
3. Your custom combination

Which works best on Rastrigin?

In [ ]:
### YOUR CODE HERE ###

# Implement different GA variants and compare!
# Use plot_convergence_comparison() to visualize results

print("Challenge: Compare operator combinations on Rastrigin!")
print("Good luck! 🚀")

---

## 📚 10. Summary

### What You Accomplished:

✅ Implemented rank-based selection  
✅ Coded Stochastic Universal Sampling  
✅ Created arithmetic and BLX-α crossover  
✅ Built polynomial and adaptive mutation  
✅ Implemented elitist replacement  
✅ Constructed a complete advanced GA  
✅ Compared operators on benchmark problems  

### Real-World Impact:

These operators are used in:
- **NSGA-II**: SBX crossover + polynomial mutation
- **CMA-ES**: Adaptive strategies
- **CHC**: BLX-α crossover
- **Industrial GAs**: Rank selection + elitism

**Next:** Move to Tutorial 3 for combinatorial optimization (TSP) and multi-objective GAs!

---

## 🎉 Congratulations!

You now master **advanced GA operators** used in production systems worldwide!

**Keep evolving!** 🧬